In [2]:
# inlegalbert_kg_rag_rrc_v3.py  (HARD-MINING + CONFUSION-GUIDED KG-RAG)
#
# Architecture:
#   InLegalBERT → BiLSTM → Multi-Head Attention Pooling → CRF
#   + Dual-Partition KG (𝒢_maj + 𝒢_rare) — GPU-resident tensors
#   + Confusion-Matrix-Guided Cross-Edges  [NEW vs v2]
#   + Weighted Focal Loss + Inverse-Freq Class Weights [NEW]
#   + Rare Document Oversampling via WeightedRandomSampler [NEW]
#   + Hard Example Replay Buffer in Phase B [NEW]
#   + Gated Graph Attention Fusion [NEW — replaces naive residual]
#   + Prototype Contrastive Auxiliary Loss [NEW]
#   + Per-class Adaptive Uncertainty Threshold [NEW]
#
# ROOT-CAUSE FIXES vs v2 (which scored LOWER than v1):
#   ① Retrieval inner Python loops killed both speed and correctness →
#      replaced with vectorised subgraph-first gather approach.
#   ② Simple residual h* = h_i + v_i let noisy KG corrupt confident
#      predictions → replaced with learned gate g ∈ (0,1).
#   ③ Label-smoothing=0.1 blurred rare class targets → reduced to 0.05
#      and replaced CE auxiliary with class-weighted Focal Loss.
#   ④ No oversampling → rare docs appear too infrequently.
#   ⑤ Confusion pairs not used → model doesn't know which majority class
#      steals from each rare class; now encoded as high-weight KG edges.
#   ⑥ Hard examples never revisited → replay buffer fills this gap.

import os, json, random, time, math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)
from sklearn.cluster import MiniBatchKMeans

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_v3_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 25          # slightly more Phase B epochs
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

LABEL_SMOOTHING  = 0.05   # ↓ from 0.1 — sharper signals for rare classes
AUX_CE_WEIGHT    = 0.25
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
WARMUP_RATIO     = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD   = 0.05

# ── KG-RAG ────────────────────────────────────────────────
KG_TOP_K        = 3
KG_TOP_NODES    = 6
KG_HOP          = 1
KG_FUSION_DIM   = 256    # SENT_OUT_DIM = 128 * 2

RST_INTRA_THRESH = 0.55  # slightly relaxed
RST_CROSS_THRESH = 0.45  # relaxed → more cross-label connections

# ── Uncertainty (per-class adaptive) ─────────────────────
UNCERTAINTY_THRESH_MAJORITY = 0.70
UNCERTAINTY_THRESH_RARE     = 0.30   # very aggressive KG use for rare

RARE_ALWAYS_KG           = True
KG_VIRTUAL_ALPHAS        = [0.25, 0.50, 0.75]
KG_PROTOTYPE_K           = 5
KG_RARE_LAMBDA           = 0.20     # retrieval priority boost for rare
KG_RARE_GAMMA            = 2.0      # r_boost multiplier in GAT
KG_MAX_VIRTUAL_PER_LABEL = 200

# ── NEW v3 hyperparameters ────────────────────────────────
FOCAL_GAMMA_RARE         = 2.5   # focal gamma for rare class sentences
FOCAL_GAMMA_MAJ          = 1.0   # focal gamma for majority class sentences
OVERSAMPLE_RARE_RATIO    = 3.0   # upsample rare-containing docs
HARD_BUFFER_SIZE         = 300   # max items in replay buffer
HARD_REPLAY_FREQ         = 4     # replay every N training steps
HARD_REPLAY_LOSS_THRESH  = 0.8   # add to buffer if loss > this
HARD_REPLAY_WEIGHT       = 0.5   # loss scale for replayed examples
PROTO_CONTRAST_WEIGHT    = 0.08  # prototype contrastive loss weight
PROTO_CONTRAST_MARGIN    = 0.35  # cosine margin (push rare proto away from maj)
CONFUSION_EDGE_WEIGHT    = 0.90  # weight of confusion-guided cross-edges
CONFUSION_TOP_K          = 3     # top confused majority classes per rare class

KG_MAX_NODES_PER_SG      = 2000
KG_MAX_CROSS_EDGES       = 4000
KG_MAX_INTRA_EDGES_STORE = 500_000

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# FOCAL LOSS  (class-weighted, per-class gamma)
# ═══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    """
    Class-weighted focal loss with per-class gamma.
    Rare classes get higher gamma (more focus on hard examples).
    Formula: FL(p_t) = -α_t · (1 - p_t)^γ_t · log(p_t)
    """
    def __init__(self, class_weights: torch.Tensor,
                 rare_ids: list,
                 gamma_rare: float = FOCAL_GAMMA_RARE,
                 gamma_maj:  float = FOCAL_GAMMA_MAJ,
                 ignore_index: int = -100):
        super().__init__()
        self.ignore_index = ignore_index
        rare_set = set(rare_ids)

        gamma_per_class = torch.full((NUM_LABELS,), gamma_maj)
        for r in rare_set:
            gamma_per_class[r] = gamma_rare

        self.register_buffer("class_weights",   class_weights.float())
        self.register_buffer("gamma_per_class", gamma_per_class)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits : (N, C), targets : (N,)
        valid = targets != self.ignore_index
        if not valid.any():
            return logits.sum() * 0.0

        logits_v  = logits[valid]
        targets_v = targets[valid]

        log_p = F.log_softmax(logits_v, dim=-1)          # (N', C)
        p     = log_p.exp()
        p_t   = p.gather(1, targets_v.unsqueeze(1)).squeeze(1)   # (N',)

        gamma_t   = self.gamma_per_class[targets_v]      # (N',)
        focal_w   = (1.0 - p_t.detach()).pow(gamma_t)    # (N',)
        class_w   = self.class_weights[targets_v]        # (N',)

        ce = F.nll_loss(log_p, targets_v, reduction="none")  # (N',)
        loss = (focal_w * class_w * ce).mean()
        return loss


def compute_class_weights(label_freqs: dict) -> torch.Tensor:
    """Inverse-frequency weights, capped at 10× for stability."""
    weights = torch.ones(NUM_LABELS)
    freqs   = [label_freqs.get(id2label[i], 1e-6) for i in range(NUM_LABELS)]
    inv     = [1.0 / max(f, 1e-6) for f in freqs]
    inv_sum = sum(inv)
    for i, w in enumerate(inv):
        weights[i] = min(w / inv_sum * NUM_LABELS, 10.0)
    return weights


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET & SAMPLER
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def build_weighted_sampler(docs: list, rare_ids: list,
                            oversample_ratio: float = OVERSAMPLE_RARE_RATIO
                            ) -> WeightedRandomSampler:
    """Upsample documents containing at least one rare-class sentence."""
    rare_set = set(rare_ids)
    weights  = []
    for _, labs in docs:
        has_rare = any(l in rare_set for l in labs)
        weights.append(oversample_ratio if has_rare else 1.0)
    w_tensor = torch.tensor(weights, dtype=torch.float)
    return WeightedRandomSampler(w_tensor,
                                  num_samples=len(w_tensor),
                                  replacement=True)


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2   # 256

        self.mha_pooling     = MultiHeadAttentionPooling(
            self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2    # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total = len(encoder_layers)
        print(f"\n❄️  BERT layers frozen: embeddings + 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)
        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)
        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids,
                      lengths=None):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids)
        sent_vecs_drop = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _ = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def _make_mask(self, emissions, labels, lengths):
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)
        return mask

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        _, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        mask = self._make_mask(emissions, labels, lengths)
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_loss = -self.crf(emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2*T2, C), labels.reshape(B2*T2))
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# GPU-ACCELERATED KNOWLEDGE GRAPH  (with confusion edges)
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    """
    GPU-resident dual-partition KG.
    Adds confusion-matrix-guided cross-edges (Phase A → B bridge).
    """

    def __init__(self, emb_dim=KG_FUSION_DIM, device=DEVICE):
        self.emb_dim        = emb_dim
        self.device         = device
        self.rare_partition = set()

        self._cpu_nodes  = defaultdict(list)  # lid → [(emb, is_virtual, is_proto)]
        self._stacked    = {}
        self._norm       = {}
        self._is_rare    = {}
        self._edge_idx   = {}
        self._edge_w     = {}
        self._proto      = {}
        self._proto_norm = {}

        # Cross-edges (parallel tensors on GPU)
        self._cx_sl = self._cx_si = None
        self._cx_dl = self._cx_di = None
        self._cx_w  = None

    # ── Population ────────────────────────────────────────
    def add_nodes(self, embeddings: torch.Tensor, label_ids: list):
        embs = embeddings.detach().cpu()
        for emb, lid in zip(embs, label_ids):
            self._cpu_nodes[lid].append((emb, False, False))

    # ── Virtual interpolation for rare classes ────────────
    def _inject_virtual(self, lid, alphas=KG_VIRTUAL_ALPHAS,
                        max_v=KG_MAX_VIRTUAL_PER_LABEL):
        real = [n[0] for n in self._cpu_nodes[lid] if not n[1] and not n[2]]
        n = len(real)
        if n < 2:
            return 0
        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        random.shuffle(pairs)
        added = 0
        for i, j in pairs:
            for alpha in alphas:
                if added >= max_v:
                    break
                v = alpha * real[i] + (1 - alpha) * real[j]
                v = F.normalize(v.unsqueeze(0), dim=-1).squeeze(0)
                self._cpu_nodes[lid].append((v, True, False))
                added += 1
            if added >= max_v:
                break
        return added

    # ── Prototype centroids ───────────────────────────────
    def _build_prototypes(self, lid, k=KG_PROTOTYPE_K):
        all_e = torch.stack([n[0] for n in self._cpu_nodes[lid]])
        n_n   = all_e.shape[0]
        k_a   = min(k, n_n)
        if k_a < 2:
            c = F.normalize(all_e.mean(0, keepdim=True), dim=-1)
        else:
            km = MiniBatchKMeans(n_clusters=k_a, n_init=5,
                                 batch_size=min(1024, n_n), random_state=SEED)
            km.fit(all_e.numpy())
            c = F.normalize(
                torch.tensor(km.cluster_centers_, dtype=torch.float32), dim=-1)
        for ci in range(c.shape[0]):
            self._cpu_nodes[lid].insert(0, (c[ci], False, True))
        return c

    # ── Main edge-build (+ optional confusion edges) ──────
    def build_edges(self,
                    rare_ids: list,
                    confusion_pairs: dict = None,
                    intra_thresh: float  = RST_INTRA_THRESH,
                    cross_thresh: float  = RST_CROSS_THRESH,
                    max_intra_per_node=5,
                    max_cross=KG_MAX_CROSS_EDGES):

        print("  Building dual-partition KG (GPU) + confusion edges ...")
        self.rare_partition = set(rare_ids)

        # ── Inject virtual nodes & prototypes for rare ────
        print("  Injecting virtual nodes & prototypes ...")
        for lid in rare_ids:
            if self._cpu_nodes[lid]:
                added = self._inject_virtual(lid)
                self._build_prototypes(lid)
                print(f"    [{id2label[lid]}] +{added} virtual | "
                      f"total={len(self._cpu_nodes[lid])} nodes")

        # ── Move all nodes to GPU ─────────────────────────
        print("  Promoting to GPU ...")
        for lid, nodes in self._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in nodes]).to(self.device)
            norms = F.normalize(embs, dim=-1)
            is_v  = torch.tensor([n[1] or n[2] for n in nodes],
                                  dtype=torch.bool, device=self.device)
            self._stacked[lid] = embs
            self._norm[lid]    = norms
            self._is_rare[lid] = is_v

        # ── Intra-label edges ─────────────────────────────
        print("  Building intra-label edges ...")
        for lid in self._stacked:
            norms = self._norm[lid]
            N = norms.shape[0]
            if N < 2:
                self._edge_idx[lid] = torch.zeros(
                    2, 0, dtype=torch.long, device=self.device)
                self._edge_w[lid]   = torch.zeros(0, device=self.device)
                continue

            if lid in self.rare_partition:
                # Dense all-pairs (chunked) for rare
                chunk = 512
                ei_l, ej_l, ew_l = [], [], []
                for start in range(0, N, chunk):
                    end = min(start + chunk, N)
                    blk = torch.mm(norms[start:end], norms.T)
                    r, c_ = torch.where(
                        (blk > 0) &
                        (torch.arange(start, end, device=self.device).unsqueeze(1)
                         < torch.arange(N, device=self.device).unsqueeze(0))
                    )
                    ei_l.append(r + start); ej_l.append(c_)
                    ew_l.append(blk[r, c_])
                    if sum(x.shape[0] for x in ei_l) >= KG_MAX_INTRA_EDGES_STORE:
                        break
                if ei_l:
                    ei = torch.cat(ei_l); ej = torch.cat(ej_l); ew = torch.cat(ew_l)
                    if ei.shape[0] > KG_MAX_INTRA_EDGES_STORE:
                        p = torch.randperm(ei.shape[0],
                                           device=self.device)[:KG_MAX_INTRA_EDGES_STORE]
                        ei, ej, ew = ei[p], ej[p], ew[p]
                    src = torch.cat([ei, ej]); dst = torch.cat([ej, ei])
                    ww  = torch.cat([ew, ew])
                else:
                    src = dst = torch.zeros(0, dtype=torch.long, device=self.device)
                    ww  = torch.zeros(0, device=self.device)
            else:
                # Sparse consecutive + high-cosine for majority
                idx  = torch.arange(N, device=self.device)
                ci   = idx[:-1]; cj = idx[1:]
                cw   = (norms[ci] * norms[cj]).sum(-1).clamp(min=0)
                sim  = torch.mm(norms, norms.T)
                sim.fill_diagonal_(-2.0)
                sim[ci, cj] = -2.0; sim[cj, ci] = -2.0
                hi_r, hi_c = torch.where(sim >= intra_thresh)
                keep = hi_r < hi_c
                hi_r, hi_c = hi_r[keep], hi_c[keep]
                hi_w = sim[hi_r, hi_c]
                if hi_r.shape[0] > N * max_intra_per_node:
                    p = torch.randperm(
                        hi_r.shape[0], device=self.device)[:N * max_intra_per_node]
                    hi_r, hi_c, hi_w = hi_r[p], hi_c[p], hi_w[p]
                ei  = torch.cat([ci, hi_r]); ej = torch.cat([cj, hi_c])
                ew  = torch.cat([cw, hi_w])
                src = torch.cat([ei, ej]); dst = torch.cat([ej, ei])
                ww  = torch.cat([ew, ew])

            self._edge_idx[lid] = torch.stack([src, dst], dim=0)
            self._edge_w[lid]   = ww

        # ── Prototype tensors ─────────────────────────────
        for lid in rare_ids:
            proto = [n for n in self._cpu_nodes.get(lid, []) if n[2]]
            if proto:
                pc = torch.stack([n[0] for n in proto]).to(self.device)
                pc = F.normalize(pc, dim=-1)
                self._proto[lid] = self._proto_norm[lid] = pc

        # ── Cross-label edges (cosine-based) ─────────────
        print("  Building cross-label edges ...")
        cx_sl, cx_si, cx_dl, cx_di, cx_w = [], [], [], [], []
        total_cross = 0
        label_ids   = sorted(self._stacked.keys())

        for a in range(len(label_ids)):
            if total_cross >= max_cross:
                break
            for b in range(a + 1, len(label_ids)):
                if total_cross >= max_cross:
                    break
                la, lb = label_ids[a], label_ids[b]
                na = self._norm[la][:KG_MAX_NODES_PER_SG]
                nb = self._norm[lb][:KG_MAX_NODES_PER_SG]
                sim = torch.mm(na, nb.T)
                rows, cols = torch.where(sim >= cross_thresh)
                rows, cols = rows[:50], cols[:50]
                if rows.shape[0] == 0:
                    continue
                w = sim[rows, cols]
                cx_sl.append(torch.full((rows.shape[0],), la,
                                        dtype=torch.long, device=self.device))
                cx_si.append(rows)
                cx_dl.append(torch.full((rows.shape[0],), lb,
                                        dtype=torch.long, device=self.device))
                cx_di.append(cols)
                cx_w.append(w)
                total_cross += rows.shape[0]

        # ── Confusion-guided edges (NEW) ──────────────────
        # For each rare class, add high-weight edges TO its most-confused
        # majority classes so GAT fusion sees the decision boundary.
        if confusion_pairs:
            print(f"  Adding confusion-guided edges "
                  f"(top-{CONFUSION_TOP_K} per rare class) ...")
            for rare_lid, confused_lids in confusion_pairs.items():
                if rare_lid not in self._stacked:
                    continue
                for clid in confused_lids[:CONFUSION_TOP_K]:
                    if clid not in self._stacked:
                        continue
                    rare_n = self._norm[rare_lid]
                    conf_n = self._norm[clid][:KG_MAX_NODES_PER_SG]
                    sim    = torch.mm(rare_n, conf_n.T)  # (R, C)
                    # Keep pairs above 0 (ensure some connectivity)
                    rows, cols = torch.where(sim > 0.3)
                    rows, cols = rows[:30], cols[:30]
                    if rows.shape[0] == 0:
                        continue
                    # Assign fixed high weight for confusion edges
                    w = torch.full((rows.shape[0],), CONFUSION_EDGE_WEIGHT,
                                   device=self.device)
                    cx_sl.append(torch.full((rows.shape[0],), rare_lid,
                                            dtype=torch.long, device=self.device))
                    cx_si.append(rows)
                    cx_dl.append(torch.full((rows.shape[0],), clid,
                                            dtype=torch.long, device=self.device))
                    cx_di.append(cols)
                    cx_w.append(w)
                    print(f"    confusion edge: {id2label[rare_lid]} → "
                          f"{id2label[clid]}  ({rows.shape[0]} pairs)")

        if cx_sl:
            self._cx_sl = torch.cat(cx_sl)
            self._cx_si = torch.cat(cx_si)
            self._cx_dl = torch.cat(cx_dl)
            self._cx_di = torch.cat(cx_di)
            self._cx_w  = torch.cat(cx_w)
        else:
            z = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cx_sl = self._cx_si = self._cx_dl = self._cx_di = z
            self._cx_w  = torch.zeros(0, device=self.device)

        n_rare_n = sum(self._stacked[l].shape[0]
                       for l in self.rare_partition if l in self._stacked)
        n_maj_n  = sum(self._stacked[l].shape[0]
                       for l in self._stacked if l not in self.rare_partition)
        n_intra  = sum(self._edge_w[l].shape[0] // 2 for l in self._edge_w)
        print(f"  KG built: maj_nodes={n_maj_n} | rare_nodes={n_rare_n}")
        print(f"           intra={n_intra} | cross={self._cx_w.shape[0]}")

    # ── Save / Load ────────────────────────────────────────
    def save(self, path):
        data = {
            "rare_partition": list(self.rare_partition),
            "nodes": {
                str(lid): [{"emb": n[0].tolist(),
                             "is_virtual": n[1], "is_proto": n[2]}
                            for n in nl]
                for lid, nl in self._cpu_nodes.items()
            },
            "intra": {str(lid): {"idx": self._edge_idx[lid].cpu().tolist(),
                                  "w":   self._edge_w[lid].cpu().tolist()}
                      for lid in self._edge_idx},
            "cross": {
                "sl": self._cx_sl.cpu().tolist() if self._cx_sl is not None else [],
                "si": self._cx_si.cpu().tolist() if self._cx_si is not None else [],
                "dl": self._cx_dl.cpu().tolist() if self._cx_dl is not None else [],
                "di": self._cx_di.cpu().tolist() if self._cx_di is not None else [],
                "w":  self._cx_w.cpu().tolist()  if self._cx_w  is not None else [],
            }
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM, device=DEVICE):
        kg = cls(emb_dim=emb_dim, device=device)
        with open(path) as f:
            data = json.load(f)
        kg.rare_partition = set(data.get("rare_partition", []))
        for k, nl in data["nodes"].items():
            lid = int(k)
            for n in nl:
                emb = torch.tensor(n["emb"], dtype=torch.float32)
                kg._cpu_nodes[lid].append((emb, n["is_virtual"], n["is_proto"]))
        for lid, nl in kg._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in nl]).to(device)
            norms = F.normalize(embs, dim=-1)
            is_v  = torch.tensor([n[1] or n[2] for n in nl],
                                  dtype=torch.bool, device=device)
            kg._stacked[lid] = embs
            kg._norm[lid]    = norms
            kg._is_rare[lid] = is_v
        for k, v in data.get("intra", {}).items():
            lid = int(k)
            kg._edge_idx[lid] = torch.tensor(
                v["idx"], dtype=torch.long,    device=device)
            kg._edge_w[lid]   = torch.tensor(
                v["w"],   dtype=torch.float32, device=device)
        cross = data.get("cross", {})
        if cross and cross.get("sl"):
            kg._cx_sl = torch.tensor(cross["sl"], dtype=torch.long,    device=device)
            kg._cx_si = torch.tensor(cross["si"], dtype=torch.long,    device=device)
            kg._cx_dl = torch.tensor(cross["dl"], dtype=torch.long,    device=device)
            kg._cx_di = torch.tensor(cross["di"], dtype=torch.long,    device=device)
            kg._cx_w  = torch.tensor(cross["w"],  dtype=torch.float32, device=device)
        else:
            z = torch.zeros(0, dtype=torch.long, device=device)
            kg._cx_sl = kg._cx_si = kg._cx_dl = kg._cx_di = z
            kg._cx_w  = torch.zeros(0, device=device)
        for lid in kg.rare_partition:
            proto = [n for n in kg._cpu_nodes.get(lid, []) if n[2]]
            if proto:
                pc = torch.stack([n[0] for n in proto]).to(device)
                pc = F.normalize(pc, dim=-1)
                kg._proto[lid] = kg._proto_norm[lid] = pc
        print(f"  KG loaded ← {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR  (per-class adaptive threshold)
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS,
                 thresh_maj=UNCERTAINTY_THRESH_MAJORITY,
                 thresh_rare=UNCERTAINTY_THRESH_RARE):
        self.log_C       = math.log(num_classes)
        self.thresh_maj  = thresh_maj
        self.thresh_rare = thresh_rare

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        H = -(probs * (probs + 1e-9).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor,
                     is_rare_pred: torch.Tensor) -> torch.Tensor:
        H = self.entropy(logits)
        thresh = torch.where(is_rare_pred,
                             torch.full_like(H, self.thresh_rare),
                             torch.full_like(H, self.thresh_maj))
        return H > thresh

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# FIXED GPU KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    """
    Vectorised retrieval — no nested Python loops over nodes.
    Strategy:
      1. Score subgraphs for ALL triggered queries in one batch mm.
      2. Select top_k subgraphs PER QUERY.
      3. For each unique selected subgraph, gather top_nodes for
         all queries that selected it  (vectorised mm, topk).
      4. 1-hop edge expansion (per-query but inner is isin on GPU).
      5. Cross-edge lookup via masked GPU tensors.
    """

    def __init__(self, kg: KnowledgeGraph, rare_ids: list,
                 top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP,
                 lambda_boost=KG_RARE_LAMBDA):
        self.kg           = kg
        self.rare_ids     = set(rare_ids)
        self.top_k        = top_k
        self.top_nodes    = top_nodes
        self.hop          = hop
        self.lambda_boost = lambda_boost
        self.device       = kg.device
        self.label_ids    = sorted(kg._stacked.keys())

    @classmethod
    def calibrate_lambda(cls, kg, rare_ids, sample_limit=200):
        rare_set  = set(rare_ids)
        maj_ids   = [l for l in kg._stacked if l not in rare_set]
        if not maj_ids:
            return KG_RARE_LAMBDA
        maj_norms = torch.cat(
            [kg._norm[m][:KG_MAX_NODES_PER_SG] for m in maj_ids if m in kg._norm],
            dim=0)
        gaps = []
        for lid in rare_ids:
            if lid not in kg._norm:
                continue
            own = kg._norm[lid]
            N   = own.shape[0]
            idx = torch.randperm(N, device=kg.device)[:sample_limit]
            s   = own[idx]
            own_sims = torch.mm(s, own.T).max(1).values
            maj_sims = torch.mm(s, maj_norms.T).max(1).values
            gaps.append((maj_sims - own_sims).cpu())
        if not gaps:
            return KG_RARE_LAMBDA
        lam = float(np.clip(torch.cat(gaps).median().item(), 0.05, 0.40))
        print(f"  λ auto-calibrated → {lam:.4f}")
        return lam

    def retrieve_batch(self, H_q: torch.Tensor,
                       trigger_mask: torch.Tensor) -> tuple:
        """
        H_q          : (T, D) L2-normalised sentence embeddings (GPU)
        trigger_mask : (T,) bool
        Returns (nb_embs, nb_weights, nb_is_rare) — all GPU, padded.
        """
        if not trigger_mask.any():
            return None, None, None

        trig_idx = trigger_mask.nonzero(as_tuple=True)[0]  # (T',)
        H_trig   = H_q[trig_idx]                           # (T', D)
        T_prime, D = H_trig.shape

        # ── Step 1: score subgraphs per query ─────────────
        sg_sims = []
        for lid in self.label_ids:
            n = self.kg._norm[lid]
            if n.shape[0] == 0:
                sg_sims.append(torch.full((T_prime,), -1.0, device=self.device))
                continue
            cap = min(n.shape[0], KG_MAX_NODES_PER_SG)
            s   = torch.mm(H_trig, n[:cap].T).max(1).values  # (T',)
            boost = self.lambda_boost if lid in self.rare_ids else 0.0
            sg_sims.append(s + boost)

        sg_score_mat = torch.stack(sg_sims, dim=1)  # (T', num_labels)
        _, topk_idx  = sg_score_mat.topk(
            min(self.top_k, len(self.label_ids)), dim=1)   # (T', top_k)

        # ── Step 2: allocate output tensors ───────────────
        cap_out = self.top_k * (self.top_nodes + 8)
        nb_embs    = torch.zeros(T_prime, cap_out, D, device=self.device)
        nb_weights = torch.zeros(T_prime, cap_out,    device=self.device)
        nb_is_rare = torch.zeros(T_prime, cap_out,
                                 dtype=torch.bool, device=self.device)
        fill = torch.zeros(T_prime, dtype=torch.long, device=self.device)

        # ── Step 3: for each unique subgraph, gather nodes ─
        unique_sg = topk_idx.unique().tolist()
        for sg_idx in unique_sg:
            lid        = self.label_ids[sg_idx]
            is_rare_sg = lid in self.rare_ids
            norms      = self.kg._norm[lid]
            embs       = self.kg._stacked[lid]
            N_sg       = norms.shape[0]
            if N_sg == 0:
                continue

            # Which queries selected this subgraph?
            uses = (topk_idx == sg_idx).any(dim=1)  # (T',)
            q_idx = uses.nonzero(as_tuple=True)[0]  # (Q,)
            H_sub = H_trig[q_idx]                   # (Q, D)

            cap_n  = min(N_sg, KG_MAX_NODES_PER_SG)
            cap_e  = embs[:cap_n]
            cap_nr = norms[:cap_n]

            # (Q, cap_n) → topk per query
            sim_mat = torch.mm(H_sub, cap_nr.T)     # (Q, cap_n)
            k_q     = min(self.top_nodes, cap_n)
            topn    = sim_mat.topk(k_q, dim=1)
            topn_idx  = topn.indices   # (Q, k_q)
            topn_sims = topn.values    # (Q, k_q)

            # ── 1-hop edge expansion ───────────────────────
            edge = self.kg._edge_idx.get(lid)

            for qi_local, qi_global in enumerate(q_idx.tolist()):
                seed = topn_idx[qi_local]          # (k_q,) GPU

                if self.hop >= 1 and edge is not None and edge.shape[1] > 0:
                    src, dst = edge[0], edge[1]
                    in_top = torch.isin(src, seed)
                    hop_n  = dst[in_top].unique()
                    if hop_n.shape[0] > 0:
                        hop_n  = hop_n[hop_n < cap_n]
                        hop_s  = torch.mv(cap_nr[hop_n], H_trig[qi_global])
                        seed   = torch.cat([seed, hop_n]).unique()

                # Write top-n nodes
                f = int(fill[qi_global].item())
                n_add = min(k_q, cap_out - f)
                if n_add <= 0:
                    continue
                use_i = topn_idx[qi_local, :n_add]
                use_s = topn_sims[qi_local, :n_add].clamp(min=0)
                nb_embs[qi_global, f:f+n_add]    = cap_e[use_i]
                nb_weights[qi_global, f:f+n_add] = use_s
                nb_is_rare[qi_global, f:f+n_add] = is_rare_sg
                fill[qi_global] = f + n_add

        # ── Step 4: cross-edge expansion ──────────────────
        cx_sl = self.kg._cx_sl
        if cx_sl is not None and cx_sl.shape[0] > 0:
            for qi_global in range(T_prime):
                f = int(fill[qi_global].item())
                if f >= cap_out:
                    continue
                # Find cross-edges where src matches any filled node
                # (simplified: check per source-label)
                # We match on label level for speed
                used_lids = topk_idx[qi_global].unique()
                for ul_idx in used_lids.tolist():
                    lid    = self.label_ids[ul_idx]
                    mask_l = (cx_sl == lid)
                    if not mask_l.any():
                        continue
                    c_di  = self.kg._cx_di[mask_l]
                    c_dl  = self.kg._cx_dl[mask_l]
                    c_w   = self.kg._cx_w[mask_l]
                    # Group by dst label
                    for dlid_t in c_dl.unique().tolist():
                        dlid = int(dlid_t)
                        dstk = self.kg._stacked.get(dlid)
                        if dstk is None:
                            continue
                        mask_d = (c_dl == dlid_t)
                        di_v   = c_di[mask_d][:3]   # max 3 cross neighbours
                        w_v    = c_w[mask_d][:3]
                        di_v   = di_v[di_v < dstk.shape[0]]
                        n_a    = min(di_v.shape[0], cap_out - f)
                        if n_a <= 0:
                            continue
                        nb_embs[qi_global, f:f+n_a]    = dstk[di_v[:n_a]]
                        nb_weights[qi_global, f:f+n_a] = w_v[:n_a]
                        nb_is_rare[qi_global, f:f+n_a] = (dlid in self.rare_ids)
                        f += n_a
                fill[qi_global] = f

        return nb_embs, nb_weights, nb_is_rare


# ═══════════════════════════════════════════════════════════
# GATED GRAPH ATTENTION FUSION  (NEW — replaces naive residual)
# ═══════════════════════════════════════════════════════════
class GatedGraphAttentionFusion(nn.Module):
    """
    Batched GAT with learned gate.

    Inputs:
        H_q     : (T', D)      – query sentence vectors
        nb_embs : (T', K, D)   – neighbour embeddings (padded)
        nb_w    : (T', K)      – edge weights (0 = padding)
        nb_rare : (T', K) bool – rare-class flags

    Output: v (T', D) — fused, gated update.

    Gate g = σ(W[H_q; v_attn]) controls how much KG context is used.
    Residual: h* = (1-g)·H_q + g·v_attn
    """

    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT,
                 r_boost=KG_RARE_GAMMA):
        super().__init__()
        self.r_boost = r_boost
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.gate    = nn.Linear(emb_dim * 2, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, H_q, nb_embs, nb_w, nb_rare):
        """Returns h* (T', D) — the gated fused representation."""
        pad_mask = (nb_w == 0)                                    # (T', K)

        q = self.proj_q(H_q)                                      # (T', D)
        k = self.proj_k(nb_embs)                                   # (T', K, D)
        dot = torch.bmm(k, q.unsqueeze(-1)).squeeze(-1) * self.scale  # (T', K)

        # r_boost: amplify rare neighbours
        r_b  = torch.where(nb_rare,
                           torch.full_like(dot, self.r_boost),
                           torch.ones_like(dot))
        raw  = dot * nb_w * r_b
        raw  = raw.masked_fill(pad_mask, -1e9)

        alpha = F.softmax(raw, dim=-1)          # (T', K)
        alpha = alpha.masked_fill(pad_mask, 0.0)
        alpha = self.dropout(alpha)

        v_attn = torch.bmm(alpha.unsqueeze(1), nb_embs).squeeze(1)  # (T', D)

        # Gating: g ∈ (0,1) per dimension
        g = torch.sigmoid(self.gate(torch.cat([H_q, v_attn], dim=-1)))  # (T', D)
        h_star = (1.0 - g) * H_q + g * v_attn
        return h_star


# ═══════════════════════════════════════════════════════════
# HARD EXAMPLE REPLAY BUFFER  (NEW)
# ═══════════════════════════════════════════════════════════
class HardExampleBuffer:
    """
    Stores (loss_val, CPU-batch) tuples for high-loss rare batches.
    Samples proportional to loss magnitude for Phase B replay.
    """

    def __init__(self, max_size=HARD_BUFFER_SIZE):
        self.max_size = max_size
        self.buf: list = []   # [(loss_float, batch_cpu_tuple)]

    def update(self, loss_val: float, batch_cpu: tuple):
        self.buf.append((loss_val, batch_cpu))
        if len(self.buf) > self.max_size:
            self.buf.sort(key=lambda x: -x[0])
            self.buf = self.buf[:self.max_size // 2]

    def sample(self) -> tuple | None:
        if not self.buf:
            return None
        losses  = torch.tensor([x[0] for x in self.buf], dtype=torch.float)
        probs   = F.softmax(losses, dim=0)
        idx     = int(torch.multinomial(probs, 1).item())
        return self.buf[idx][1]

    def __len__(self):
        return len(self.buf)


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL  (gated fusion + prototype contrastive)
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):

    def __init__(self, base_model: InLegalBERT_BiLSTM_MHA_CRF,
                 kg: KnowledgeGraph,
                 rare_ids: list,
                 retriever: KGRetriever = None,
                 focal_loss: FocalLoss  = None):
        super().__init__()
        self.base       = base_model
        self.kg         = kg
        self.rare_ids   = set(rare_ids)
        self.rare_list  = sorted(rare_ids)
        self.retriever  = retriever or KGRetriever(kg, rare_ids)
        self.uncertainty = UncertaintyEstimator()
        self.focal_loss  = focal_loss  # may be None

        sent_dim = base_model.sent_out_dim  # 256
        ctx_dim  = base_model.ctx_out_dim   # 128

        self.gat_fusion = GatedGraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        self._rare_t = torch.tensor(self.rare_list, dtype=torch.long)

    def _rare_mask(self, top_labels):
        rare = self._rare_t.to(top_labels.device)
        return (top_labels.unsqueeze(-1) == rare.view(1, 1, -1)).any(-1)

    # ── Prototype contrastive loss ─────────────────────────
    def _proto_contrast_loss(self, device) -> torch.Tensor:
        """
        Push rare prototypes away from majority prototypes.
        Loss = mean max(0, sim(rare_p, maj_p) - margin)
        """
        rare_protos = [self.kg._proto_norm[l] for l in self.rare_list
                       if l in self.kg._proto_norm]
        if not rare_protos:
            return torch.tensor(0.0, device=device)

        maj_ids    = [l for l in self.kg._stacked if l not in self.rare_ids]
        maj_protos = [self.kg._proto_norm[l] for l in maj_ids
                      if l in self.kg._proto_norm]
        if not maj_protos:
            return torch.tensor(0.0, device=device)

        rp = F.normalize(torch.cat(rare_protos, dim=0), dim=-1).to(device)  # (R, D)
        mp = F.normalize(torch.cat(maj_protos,  dim=0), dim=-1).to(device)  # (M, D)
        sim = torch.mm(rp, mp.T)     # (R, M)
        loss = torch.clamp(sim - PROTO_CONTRAST_MARGIN, min=0.0).mean()
        return loss

    # ── Vectorised KG fusion ──────────────────────────────
    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, D = sent_vecs.shape
        fused       = sent_vecs.clone()
        top_labels  = self.uncertainty.top_label(emissions)    # (B, T)
        is_rare_p   = self._rare_mask(top_labels)              # (B, T)
        uncertain   = self.uncertainty.is_uncertain(emissions, is_rare_p)
        trigger     = uncertain | (RARE_ALWAYS_KG & is_rare_p)

        for b in range(B):
            n      = int(lengths[b].item())
            trig_b = trigger[b, :n]
            if not trig_b.any():
                continue

            H_b      = sent_vecs[b, :n]                           # (n, D)
            H_b_norm = F.normalize(H_b.detach(), dim=-1)          # for retrieval

            nb_embs, nb_w, nb_rare = self.retriever.retrieve_batch(
                H_b_norm, trig_b)
            if nb_embs is None:
                continue

            trig_idx = trig_b.nonzero(as_tuple=True)[0]           # (T',)
            H_q      = H_b[trig_idx]                              # (T', D)

            # Gated GAT fusion
            h_star = self.gat_fusion(H_q, nb_embs, nb_w, nb_rare)  # (T', D)
            fused[b, trig_idx] = h_star  # gate handles blending internally

        return fused

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        device = input_ids.device

        # Step 1: base first pass
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        # Step 2: KG fusion
        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device)

        # Step 3: re-run ctx-BiLSTM on fused representations
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)
        fused_ctx = self.base.dropout(fused_ctx)

        # Step 4: fusion classifier
        fused_emis = self.fusion_classifier(fused_ctx)
        fused_emis = torch.nan_to_num(fused_emis, nan=0.0, posinf=1e4, neginf=-1e4)

        # Mask
        if lengths is not None:
            B, T, _ = fused_emis.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emis.shape[:2], dtype=torch.bool, device=device)

        combined = (base_emissions + fused_emis) / 2.0

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0

            base_crf  = -self.base.crf(
                base_emissions, safe, mask=mask, reduction="mean")
            fused_crf = -self.fusion_crf(
                fused_emis,     safe, mask=mask, reduction="mean")

            B2, T2, C = combined.shape
            flat_logits = combined.reshape(B2*T2, C)
            flat_labels = labels.reshape(B2*T2)

            if self.focal_loss is not None:
                aux_loss = self.focal_loss(flat_logits, flat_labels)
            else:
                aux_loss = self.ce_loss(flat_logits, flat_labels)

            proto_loss = self._proto_contrast_loss(device)
            loss = ((base_crf + fused_crf) / 2.0
                    + AUX_CE_WEIGHT * aux_loss
                    + PROTO_CONTRAST_WEIGHT * proto_loss)
            return loss, combined
        else:
            decoded = self.fusion_crf.decode(fused_emis, mask=mask)
            return decoded, fused_emis


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds,
                                     labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds,
                                   labels=list(range(NUM_LABELS)),
                                   average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {"f1": float(per_class_f1[i]),
                      "precision": float(per_class_prec[i]),
                      "recall":    float(per_class_rec[i])}
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_t  = [id2label[x] for x in all_trues]
    str_p  = [id2label[x] for x in all_preds]
    cls_rp = classification_report(str_t, str_p, labels=LABELS,
                                    digits=4, zero_division=0)
    cm     = confusion_matrix(str_t, str_p, labels=LABELS)

    return dict(
        macro_f1=macro_f1, micro_f1=micro_f1, weighted_f1=weighted_f1,
        macro_precision=macro_prec, micro_precision=micro_prec,
        weighted_precision=weighted_prec,
        macro_recall=macro_rec, micro_recall=micro_rec,
        weighted_recall=weighted_rec,
        rare_f1=rare_f1, rare_precision=rare_prec, rare_recall=rare_rec,
        per_class_metrics=per_class_metrics,
        accuracy=acc, cls_report=cls_rp, cm=cm,
        all_preds=all_preds, all_trues=all_trues,
    )


def count_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if     p.requires_grad)
    fr = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {tr:,} | Frozen: {fr:,}")
    return tr, fr


# ═══════════════════════════════════════════════════════════
# CONFUSION-PAIR COMPUTATION  (Phase A → KG bridge)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def compute_confusion_pairs(model, dataset, rare_ids,
                             device=DEVICE, top_k=CONFUSION_TOP_K) -> dict:
    """
    Evaluate base model on train set. For each rare class r, find
    the top-K majority classes that predictions most confuse it with.
    Returns: { rare_lid: [confused_lid, ...] }
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=2, shuffle=False,
                        collate_fn=collate_rrc)
    all_preds, all_trues = [], []
    rare_set = set(rare_ids)

    for ids, attn, ttype, labels, lengths in loader:
        ids     = ids.to(device)
        attn    = attn.to(device)
        ttype   = ttype.to(device)
        lengths = lengths.to(device)
        decoded, _ = model(ids, attn, ttype, labels=None, lengths=lengths)
        for i, seq_p in enumerate(decoded):
            true_len = int(lengths[i].item())
            all_preds.extend(seq_p)
            all_trues.extend(labels[i, :true_len].tolist())

    confusion = defaultdict(Counter)
    for true, pred in zip(all_trues, all_preds):
        if true in rare_set and pred != true:
            confusion[true][pred] += 1

    result = {}
    print("\n📊 Confusion-guided edge analysis:")
    for rid, counter in confusion.items():
        top_confused = [cls for cls, _ in counter.most_common(top_k)]
        result[rid] = top_confused
        names = [id2label[c] for c in top_confused]
        total = sum(counter.values())
        print(f"  {id2label[rid]:<20} confused → {names}  "
              f"(total misclassified: {total})")

    # Also print per-rare CM summary from sklearn
    str_t = [id2label[x] for x in all_trues]
    str_p = [id2label[x] for x in all_preds]
    rare_labels = [id2label[r] for r in rare_ids]
    cm = confusion_matrix(str_t, str_p, labels=LABELS)
    print("\n  Full confusion matrix saved for analysis.")

    return result


# ═══════════════════════════════════════════════════════════
# BASE TRAINER  (Phase A — with focal loss + oversampling)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, focal_loss: FocalLoss = None, device=DEVICE):
        self.model      = model.to(device)
        self.focal_loss = focal_loss
        self.device     = device

    def build_optimizer(self):
        pg = []
        pg.append({"params": list(self.model.bert.pooler.parameters()),
                   "lr": BERT_LR, "weight_decay": WEIGHT_DECAY})
        enc = self.model.bert.encoder.layer
        n   = len(enc)
        for i in range(n - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in enc[i].parameters() if p.requires_grad]
            if params:
                pg.append({"params": params, "lr": lr_i,
                           "weight_decay": WEIGHT_DECAY})
        head_mods = [self.model.sent_bilstm, self.model.mha_pooling,
                     self.model.sent_layer_norm, self.model.ctx_bilstm,
                     self.model.classifier, self.model.crf]
        head_p = [p for m in head_mods for p in m.parameters()]
        pg.append({"params": head_p, "lr": HEAD_LR,
                   "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(pg)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                tt  = tt.to(self.device);  labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, tt,
                                     labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total += loss.item(); n += 1
        return total / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, tt,
                                        labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :tl].tolist())

        if measure_inference_time:
            ti  = time.time() - t0
            ns  = len(all_trues)
            inf = {"total_inference_time_s": ti,
                   "latency_per_document_ms": ti / max(1, n_samples) * 1000,
                   "throughput_sentences_per_s": ns / max(1e-9, ti)}
            with open(os.path.join(OUT_DIR,
                                   f"inference_{split_name}.json"), "w") as f:
                json.dump(inf, f, indent=2)
        else:
            inf = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if inf:
            metrics["inference_time_info"] = inf
        return metrics

    def train(self, train_docs, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):

        # Oversampling via WeightedRandomSampler
        sampler      = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)

        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)

        early_stopper = EarlyStopping()
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            t0 = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, tt, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  labels = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, emissions = self.model(
                    ids, attn, tt, labels=labels, lengths=lengths)

                # Add focal loss auxiliary on top of base CRF loss
                if self.focal_loss is not None:
                    B2, T2, C = emissions.shape
                    fl = self.focal_loss(
                        emissions.reshape(B2*T2, C).detach(),
                        labels.reshape(B2*T2))
                    loss = loss + AUX_CE_WEIGHT * fl

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                run_loss += loss.item(); n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            ep_t     = time.time() - t0
            avg_loss = run_loss / max(1, n_steps)
            val_loss = self.compute_val_loss(dev_dataset)
            val_m    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Ep {epoch:03d}/{num_epochs} | "
                f"tr_loss:{avg_loss:.4f} val_loss:{val_loss:.4f} | "
                f"mac_F1:{val_m['macro_f1']:.4f} rare_F1:{val_m['rare_f1']:.4f} | "
                f"t:{ep_t:.1f}s ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_loss, "val_loss": val_loss,
                "val_macro_f1": val_m["macro_f1"],
                "val_micro_f1": val_m["micro_f1"],
                "val_weighted_f1": val_m["weighted_f1"],
                "val_rare_f1": val_m["rare_f1"],
                "val_accuracy": val_m["accuracy"],
                "epoch_train_time_s": ep_t,
            })

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_m["macro_f1"]):
                print(f"\n⏹  Base early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "base_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")

        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))
            print(f"  Base model saved to {BEST_MODEL_DIR}/base_model.bin")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer,
                           rare_ids, confusion_pairs, device=DEVICE) -> KnowledgeGraph:
    print("\n🔨 Building Dual-Partition KG + Confusion Edges ...")
    base_model.eval(); base_model.to(device)

    kg     = KnowledgeGraph(emb_dim=base_model.sent_out_dim, device=device)
    dummy  = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy, batch_size=4, shuffle=False,
                        collate_fn=collate_rrc, num_workers=0)

    for doc_idx, (ids, attn, tt, labels, lengths) in enumerate(loader):
        ids  = ids.to(device); attn = attn.to(device); tt = tt.to(device)
        for bi in range(ids.shape[0]):
            sv = base_model.encode_sentences(
                ids[bi:bi+1], attn[bi:bi+1], tt[bi:bi+1]).squeeze(0)
            n  = int(lengths[bi].item())
            kg.add_nodes(sv[:n].cpu(), labels[bi, :n].tolist())
        if (doc_idx + 1) % 50 == 0:
            print(f"  {(doc_idx+1)*ids.shape[0]}/{len(train_docs)} docs processed")

    kg.build_edges(rare_ids=rare_ids, confusion_pairs=confusion_pairs)
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER  (Phase B — with hard example replay)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: KGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_p = (list(self.model.gat_fusion.parameters()) +
                 list(self.model.fusion_classifier.parameters()) +
                 list(self.model.fusion_crf.parameters()))
        base_p = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_p,  "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_p, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, tt, labels, lengths in loader:
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, tt,
                                        labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :tl].tolist())
        if measure_inference_time:
            ti  = time.time() - t0
            ns  = len(all_trues)
            inf = {"total_inference_time_s": ti,
                   "latency_per_document_ms": ti / max(1, n_samples) * 1000,
                   "throughput_sentences_per_s": ns / max(1e-9, ti)}
            with open(os.path.join(OUT_DIR,
                                   f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(inf, f, indent=2)
        else:
            inf = None
        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if inf:
            metrics["inference_time_info"] = inf
        return metrics

    def train(self, train_docs, train_dataset, dev_dataset,
              rare_ids, num_epochs=NUM_EPOCHS_KG):
        # Oversampling of rare docs in Phase B as well
        sampler      = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)

        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)

        early_stopper = EarlyStopping(patience=7)
        buffer        = HardExampleBuffer(max_size=HARD_BUFFER_SIZE)
        rare_set      = set(rare_ids)
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            t0 = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, tt, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device); attn = attn.to(self.device)
                tt      = tt.to(self.device);  labels = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, tt,
                                     labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                run_loss += loss.item(); n_steps += 1

                # ── Hard example buffering ─────────────────
                lv = loss.item()
                if lv > HARD_REPLAY_LOSS_THRESH:
                    flat_l = labels.cpu().flatten()
                    has_rare = any(int(x) in rare_set
                                   for x in flat_l if x.item() >= 0)
                    if has_rare:
                        buffer.update(lv, (ids.cpu(), attn.cpu(),
                                           tt.cpu(), labels.cpu(),
                                           lengths.cpu()))

                # ── Hard example replay ────────────────────
                if (step + 1) % HARD_REPLAY_FREQ == 0 and len(buffer) >= 10:
                    replay = buffer.sample()
                    if replay:
                        r_ids, r_attn, r_tt, r_lab, r_len = [
                            x.to(self.device) for x in replay]
                        r_loss, _ = self.model(r_ids, r_attn, r_tt,
                                               labels=r_lab, lengths=r_len)
                        if not torch.isnan(r_loss) and not torch.isinf(r_loss):
                            (r_loss * HARD_REPLAY_WEIGHT).backward()
                            torch.nn.utils.clip_grad_norm_(
                                self.model.parameters(), GRAD_CLIP)
                            optimizer.step(); scheduler.step(); optimizer.zero_grad()

            ep_t     = time.time() - t0
            avg_loss = run_loss / max(1, n_steps)
            val_m    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG]  Ep {epoch:02d}/{num_epochs} | "
                f"tr_loss:{avg_loss:.4f} | "
                f"mac_F1:{val_m['macro_f1']:.4f} "
                f"rare_F1:{val_m['rare_f1']:.4f} | "
                f"t:{ep_t:.1f}s buf:{len(buffer)} "
                f"ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "kg",
                "train_loss": avg_loss,
                "val_macro_f1": val_m["macro_f1"],
                "val_rare_f1":  val_m["rare_f1"],
                "val_accuracy": val_m["accuracy"],
                "epoch_train_time_s": ep_t,
            })

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_m["macro_f1"]):
                print(f"\n⏹  KG early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "kg_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS,
                yticklabels=LABELS, cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (KG-RAG v3)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1  (KG-RAG v3)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"],
            marker="o", markersize=3, label="Train Loss")
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Combined Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)
    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1",  marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1  (Base → KG-RAG v3)"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def print_metrics_table(dev_m, test_m, base_t=None, kg_t=None, n_params=None):
    rows = [
        ("Accuracy",          "accuracy"),
        ("Macro-F1",          "macro_f1"),
        ("Micro-F1",          "micro_f1"),
        ("Weighted-F1",       "weighted_f1"),
        ("Minority Macro-F1", "rare_f1"),
        ("Macro-Precision",   "macro_precision"),
        ("Macro-Recall",      "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  — InLegalBERT+BiLSTM+MHA+CRF+KG-RAG v3")
    print("=" * 72)
    if n_params: print(f"  Trainable params : {n_params:,}")
    if base_t:   print(f"  Phase A time     : {base_t/60:.1f} min")
    if kg_t:     print(f"  Phase B time     : {kg_t/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<30} {'Dev':>10} {'Test':>10}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<30} {dev_m[key]:>10.4f} {test_m[key]:>10.4f}")
    print("=" * 72)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 66)
    print(f"  {'Label':<22} {'F1-Dev':>8} {'F1-Test':>8} "
          f"{'Prec':>8} {'Rec':>8}")
    print("  " + "-" * 66)
    for lbl in LABELS:
        dv = dev_m["per_class_metrics"][lbl]
        ts = test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>8.4f} {ts['f1']:>8.4f} "
              f"{ts['precision']:>8.4f} {ts['recall']:>8.4f}")
    print("  " + "-" * 66)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT+BiLSTM+MHA+CRF → KG-RAG v3")
    print("  [+] Focal Loss  [+] Oversampling  [+] Confusion Edges")
    print("  [+] Hard Mining  [+] Gated Fusion  [+] Prototype Contrast\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train:{len(train_docs)} | Dev:{len(dev_docs)} | Test:{len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    pd.DataFrame([{"label": l, "frequency": label_freqs[l],
                   "is_rare": l in rare_labels}
                  for l in LABELS]
                 ).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    # ── Focal loss + class weights ─────────────────────────
    class_weights = compute_class_weights(label_freqs)
    focal_loss    = FocalLoss(class_weights, rare_ids).to(DEVICE)
    print(f"\nClass weights (top-5 rare): "
          f"{[(id2label[r], round(float(class_weights[r]),2)) for r in rare_ids[:5]]}")

    print("Loading tokenizer ...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ══════════════════════════════════════════════════════
    # PHASE A: Train base model
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training (focal loss + oversampling)")
    print("=" * 60)

    base_model   = InLegalBERT_BiLSTM_MHA_CRF()
    base_trainer = BaseTrainer(base_model, focal_loss=focal_loss, device=DEVICE)
    base_hist, base_time = base_trainer.train(
        train_docs, train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE)

    # ══════════════════════════════════════════════════════
    # PHASE A→B: Confusion-guided KG construction
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Confusion-Matrix Analysis + KG Construction")
    print("=" * 60)

    # Compute which majority classes each rare class is confused with
    confusion_pairs = compute_confusion_pairs(
        base_model, train_dataset, rare_ids, device=DEVICE)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG — loading ...")
        kg = KnowledgeGraph.load(
            kg_path, emb_dim=base_model.sent_out_dim, device=DEVICE)
        kg.rare_partition = set(rare_ids)
    else:
        kg = build_knowledge_graph(
            base_model, train_docs, tokenizer,
            rare_ids=rare_ids, confusion_pairs=confusion_pairs, device=DEVICE)

    # Auto-calibrate λ
    print("\n  Calibrating λ ...")
    lambda_boost = KGRetriever.calibrate_lambda(kg, rare_ids)

    # ══════════════════════════════════════════════════════
    # PHASE B: KG-augmented fine-tuning with hard mining
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: KG-Augmented Fine-Tuning + Hard Example Replay")
    print(f"  λ={lambda_boost:.4f}  γ={KG_RARE_GAMMA}  "
          f"focal_γ_rare={FOCAL_GAMMA_RARE}  "
          f"thresh_rare={UNCERTAINTY_THRESH_RARE}")
    print("=" * 60)

    retriever = KGRetriever(
        kg, rare_ids=rare_ids,
        top_k=KG_TOP_K, top_nodes=KG_TOP_NODES,
        hop=KG_HOP, lambda_boost=lambda_boost)

    kg_model = KGAugmentedModel(
        base_model=base_model, kg=kg,
        rare_ids=rare_ids, retriever=retriever,
        focal_loss=focal_loss)

    n_params, _ = count_parameters(kg_model)

    kg_trainer = KGTrainer(kg_model, device=DEVICE)
    kg_hist, kg_time = kg_trainer.train(
        train_docs, train_dataset, dev_dataset,
        rare_ids=rare_ids, num_epochs=NUM_EPOCHS_KG)

    plot_combined_history(base_hist, kg_hist)

    # ══════════════════════════════════════════════════════
    # EVALUATION
    # ══════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_m = kg_trainer.evaluate(dev_dataset, rare_ids,
                                split_name="dev",
                                measure_inference_time=True)
    print(f"  Dev  Accuracy:{dev_m['accuracy']:.4f} "
          f"Macro-F1:{dev_m['macro_f1']:.4f} "
          f"Rare-F1:{dev_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF+KG-RAG v3\n")
        f.write(f"Techniques: FocalLoss, Oversampling, ConfusionEdges, "
                f"HardMining, GatedFusion, ProtoContrast\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_m["cls_report"])

    save_confusion_matrix(dev_m["cm"],  "dev",  rare_labels)
    save_per_class_f1_chart(dev_m["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_m = kg_trainer.evaluate(test_dataset, rare_ids,
                                 split_name="test",
                                 measure_inference_time=True)
    print(f"  Test Accuracy:{test_m['accuracy']:.4f} "
          f"Macro-F1:{test_m['macro_f1']:.4f} "
          f"Rare-F1:{test_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF+KG-RAG v3\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_m["cls_report"])

    save_confusion_matrix(test_m["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_m["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_m["all_trues"]],
        "pred": [id2label[x] for x in test_m["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall",    "micro_recall",    "weighted_recall",    "rare_recall",
        "accuracy",
    ]
    summary = {
        "model": "InLegalBERT+BiLSTM+MHA+CRF+KG-RAG-v3",
        "improvements": [
            "FocalLoss(gamma_rare=2.5,gamma_maj=1.0)+InvFreqWeights",
            "WeightedRandomSampler(ratio=3.0)",
            "ConfusionMatrixGuidedEdges",
            "HardExampleReplayBuffer",
            "GatedGraphAttentionFusion",
            "PrototypeContrastiveLoss",
            "AdaptiveUncertaintyThreshold",
        ],
        "kg_config": {
            "lambda_boost":  lambda_boost,
            "r_boost_gamma": KG_RARE_GAMMA,
            "proto_k":       KG_PROTOTYPE_K,
            "virtual_alphas": KG_VIRTUAL_ALPHAS,
            "thresh_rare":   UNCERTAINTY_THRESH_RARE,
            "thresh_maj":    UNCERTAINTY_THRESH_MAJORITY,
            "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES, "hop": KG_HOP,
        },
        "timing": {"phase_a_s": base_time, "phase_b_s": kg_time,
                   "total_s": base_time + kg_time},
        "rare_classes": rare_labels,
        "dev":  {k: dev_m[k]  for k in scalar_keys},
        "test": {k: test_m[k] for k in scalar_keys},
        "per_class_dev":  dev_m["per_class_metrics"],
        "per_class_test": test_m["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_metrics_table(dev_m, test_m,
                        base_t=base_time, kg_t=kg_time, n_params=n_params)
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT+BiLSTM+MHA+CRF → KG-RAG v3
  [+] Focal Loss  [+] Oversampling  [+] Confusion Edges
  [+] Hard Mining  [+] Gated Fusion  [+] Prototype Contrast

Loading JSONL files ...
  Train:245 | Dev:30 | Test:50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + 0-7.
🔥 BERT trainable: 8-11 + pooler.

[Base] Ep 001/60 | tr_loss:284.8021 val_loss:211.6492 | mac_F1:0.0391 rare_F1:0.0000 | t:61.2s ES:0/10
  ✔ New best val_macro_f1=0.0391
[Base] Ep 002/60 | tr_loss:231.5278 val_loss:171.5741 | mac_F1:0.0727 rare_F1:0.0000 | t:62.5s ES:0/10
  ✔ New best val_macro_f1=0.0727
[Base] Ep 003/60 | tr_loss:190.3178 val_loss:131.4388 | mac_F1:0.2161 rare_F1:0.0598 | t:59.0s ES:0/10
  ✔ New best val_macro_f1=0.2161
[Base] Ep 004/60 | tr_loss:153.1226 val_loss:100.8132 | mac_F1:0.2719 rare_F1:0.1128 | t:62.5s ES:0/10
  ✔ New best val_macro_f1=0.2719
[Base] Ep 005/60 | tr_loss:141.0998 val_loss:93.3196 | mac_F1:0.2704 rare_F1:0.1118 | t:65.2s ES:0/10
[Base] Ep 006/60 | tr_loss:129.2616 val_loss:86.8265 | mac_F1:0.2998 rare_F1:0.1419 | t:84.6s ES:1/10
  ✔ New best val_macro_f1=0.2998
[Base] Ep 007/60 | tr_loss:120.6087 val_loss:80.5756 | mac_F1:0.3316 rare_F1:0.1776 | t:70.6s ES:0/10
  ✔ New best val_macro_f1=0.3316
[Base] 